## What is a Markov Decision Process?

An **MDP** is a mathematical framework for modeling sequential decision-making:

$$\text{MDP} = \langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$$

Where:
- $\mathcal{S}$: State space
- $\mathcal{A}$: Action space  
- $P(s'|s,a)$: Transition dynamics
- $R(s,a,s')$: Reward function
- $\gamma \in [0,1]$: Discount factor

<div style="background-color: #f0f0f5; padding: 15px; border-left: 5px solid #1c7293; margin-top: 20px;">
    <strong>Key Property:</strong> The Markov Property states that the future depends only on the current state, not the history:
    $$P(s_{t+1}|s_t, a_t, s_{t-1}, a_{t-1}, \ldots) = P(s_{t+1}|s_t, a_t)$$
</div>

# Module 2: MDP Fundamentals
## Interactive Demonstration with CartPole

<div style="text-align: center; padding: 20px; background-color: #065a82; color: white; border-radius: 10px;">
    <h3>Reinforcement Learning for EDA</h3>
    <p>Spring 2025 | Week 2</p>
</div>

## Today's Example: CartPole

<div style="display: flex; gap: 30px;">
<div style="flex: 1;">
    
**The Challenge:**
Balance a pole on a moving cart by applying left/right forces.

**Why CartPole?**
- Simple to visualize
- Clear MDP structure
- Demonstrates key RL concepts
- Analogous to EDA decision problems

</div>
<div style="flex: 1; background-color: #e8f4f8; padding: 20px; border-radius: 10px;">
    
**Success Criteria:**
- Keep pole angle < 12°
- Keep cart within bounds
- Maximize episode length
- Episode ends at 500 steps or failure

</div>
</div>

In [2]:
# Setup: Import required libraries
import gymnasium as gym
# Quick training of DQN (simplified)
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.env_util import make_vec_env
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [3]:
# Animation helper function
def create_cartpole_animation(policy_fn, policy_name, seed=42, max_frames=200):
    """Create animation of CartPole episode."""
    env = gym.make('CartPole-v1', render_mode='rgb_array')
    state, _ = env.reset(seed=seed)
    
    frames = []
    total_reward = 0
    
    for step in range(max_frames):
        # Render current frame
        frame = env.render()
        frames.append(frame)
        
        # Get action from policy
        if policy_name == 'random':
            action = env.action_space.sample()
        elif policy_name == 'heuristic':
            action = policy_fn(state)
        else:  # trained
            action, _ = policy_fn.predict(state, deterministic=True)
        
        # Take step
        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        
        if terminated or truncated:
            # Add a few frames at the end
            for _ in range(10):
                frames.append(env.render())
            break
    
    env.close()
    
    # Create animation
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('off')
    
    img = ax.imshow(frames[0])
    
    def animate(frame_idx):
        img.set_array(frames[frame_idx])
        return [img]
    
    anim = animation.FuncAnimation(
        fig, animate, frames=len(frames), 
        interval=50, blit=True, repeat=True
    )
    
    plt.close(fig)
    
    return anim, total_reward, len(frames)

## MDP Component 1: State Space $\mathcal{S}$

The **state** contains all information needed to make decisions.

### CartPole State: 4-dimensional continuous vector

| Dimension | Description | Range |
|-----------|-------------|-------|
| $s_0$ | Cart position | $[-4.8, 4.8]$ |
| $s_1$ | Cart velocity | $[-\infty, \infty]$ |
| $s_2$ | Pole angle | $[-0.418, 0.418]$ rad ($\approx \pm 24°$) |
| $s_3$ | Pole angular velocity | $[-\infty, \infty]$ |

<div style="background-color: #fff3cd; padding: 15px; border-left: 5px solid #ffc107; margin-top: 20px;">
    <strong>⚡ EDA Connection:</strong> Like choosing which variable to branch on in SAT solving, or which gate to map next - the state captures the current configuration and progress.
</div>

In [4]:
# Let's examine the state space
env = gym.make('CartPole-v1')
state, info = env.reset(seed=42)

print("Initial State:")
print(f"  Cart position:        {state[0]:>8.4f} (bounds: ±4.8)")
print(f"  Cart velocity:        {state[1]:>8.4f}")
print(f"  Pole angle:           {state[2]:>8.4f} rad ({np.degrees(state[2]):>6.2f}°)")
print(f"  Pole angular velocity:{state[3]:>8.4f}")
print(f"\nState shape: {state.shape}")
print(f"State space: {env.observation_space}")

Initial State:
  Cart position:          0.0274 (bounds: ±4.8)
  Cart velocity:         -0.0061
  Pole angle:             0.0359 rad (  2.05°)
  Pole angular velocity:  0.0197

State shape: (4,)
State space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)


## MDP Component 2: Action Space $\mathcal{A}$

**Actions** are the decisions available to the agent at each state.

### CartPole Actions: 2 discrete choices

<div style="display: flex; gap: 20px; margin-top: 20px;">
<div style="flex: 1; background-color: #e3f2fd; padding: 20px; border-radius: 10px; text-align: center;">
    <h4 style="color: #065a82;">Action 0</h4>
    <p style="font-size: 1.2em;">← Push cart LEFT</p>
</div>
<div style="flex: 1; background-color: #e8f5e9; padding: 20px; border-radius: 10px; text-align: center;">
    <h4 style="color: #1c7293;">Action 1</h4>
    <p style="font-size: 1.2em;">Push cart RIGHT →</p>
</div>
</div>

$$\mathcal{A} = \{0, 1\}$$

<div style="background-color: #fff3cd; padding: 15px; border-left: 5px solid #ffc107; margin-top: 20px;">
    <strong>⚡ EDA Connection:</strong> Like binary decisions in placement (left/right partition), technology mapping (use/skip a gate), or SAT (set variable true/false).
</div>

In [5]:
# Examine action space
print(f"Action space: {env.action_space}")
print(f"Number of actions: {env.action_space.n}")
print(f"\nValid actions: {list(range(env.action_space.n))}")
print("  0 = Push LEFT")
print("  1 = Push RIGHT")

Action space: Discrete(2)
Number of actions: 2

Valid actions: [0, 1]
  0 = Push LEFT
  1 = Push RIGHT


## MDP Component 3: Reward Function $R$

**Rewards** provide feedback on action quality.

### CartPole Reward Structure:

$$R(s, a, s') = \begin{cases}
+1 & \text{if episode continues} \\
0 & \text{if episode terminates (failure)}
\end{cases}$$

**Episode terminates when:**
- Pole angle > 12° (pole fell)
- Cart position outside $[-2.4, 2.4]$ (hit boundary)
- 500 steps reached (success!)

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-top: 20px;">
    <strong>Goal:</strong> Maximize cumulative reward = Maximize episode length
    $$G_t = \sum_{k=0}^{T} \gamma^k R_{t+k+1}$$
</div>

In [7]:
# Demonstrate reward accumulation
env = gym.make('CartPole-v1')
state, _ = env.reset(seed=42)

total_reward = 0
for step in range(10):
    action = env.action_space.sample()  # Random action
    next_state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    
    action_name = "LEFT" if action == 0 else "RIGHT"
    print(f"Step {step+1}: Action={action_name}, Reward={reward:.0f}, Total={total_reward:.0f}")
    
    if terminated or truncated:
        print(f"\n❌ Episode ended after {step+1} steps")
        break
    state = next_state
else:
    print(f"\n✓ Episode still running after 10 steps")

env.close()

Step 1: Action=LEFT, Reward=1, Total=1
Step 2: Action=LEFT, Reward=1, Total=2
Step 3: Action=RIGHT, Reward=1, Total=3
Step 4: Action=LEFT, Reward=1, Total=4
Step 5: Action=RIGHT, Reward=1, Total=5
Step 6: Action=RIGHT, Reward=1, Total=6
Step 7: Action=LEFT, Reward=1, Total=7
Step 8: Action=RIGHT, Reward=1, Total=8
Step 9: Action=RIGHT, Reward=1, Total=9
Step 10: Action=RIGHT, Reward=1, Total=10

✓ Episode still running after 10 steps


## MDP Component 4: Transition Dynamics $P$

**Transitions** define how actions change states.

$$P(s_{t+1} | s_t, a_t) = \text{Probability of reaching } s_{t+1} \text{ from } s_t \text{ via action } a_t$$

### CartPole Physics (Deterministic):

```
Force = +10N (right) or -10N (left)
Cart mass = 1.0 kg
Pole mass = 0.1 kg  
Pole length = 0.5 m
Timestep = 0.02 s
```

The next state is **deterministically** computed using physics equations (pendulum dynamics).

<div style="background-color: #fff3cd; padding: 15px; border-left: 5px solid #ffc107; margin-top: 20px;">
    <strong>⚡ EDA Connection:</strong> Like how placing a gate affects remaining placement options, or how setting a variable constrains the SAT search space.
</div>

In [8]:
# Demonstrate state transitions
env = gym.make('CartPole-v1')
state, _ = env.reset(seed=42)

print("Transition Example:")
print(f"Initial state: {state}")
print(f"  Pole angle: {np.degrees(state[2]):.2f}°\n")

# Apply action 1 (push right)
next_state, reward, terminated, truncated, info = env.step(1)
print(f"After pushing RIGHT:")
print(f"Next state: {next_state}")
print(f"  Pole angle: {np.degrees(next_state[2]):.2f}°")
print(f"  Cart moved: {next_state[0] - state[0]:.4f} units")
print(f"  Reward: {reward}")

env.close()

Transition Example:
Initial state: [ 0.0273956  -0.00611216  0.03585979  0.0197368 ]
  Pole angle: 2.05°

After pushing RIGHT:
Next state: [ 0.02727336  0.18847767  0.03625453 -0.26141977]
  Pole angle: 2.08°
  Cart moved: -0.0001 units
  Reward: 1.0


## Policy $\pi$: The Decision-Maker

A **policy** is a mapping from states to actions:

$$\pi: \mathcal{S} \rightarrow \mathcal{A}$$

Or probabilistically:

$$\pi(a|s) = P(A_t = a | S_t = s)$$

### Three Policy Examples:

1. **Random Policy**: Choose actions uniformly at random
2. **Heuristic Policy**: Use domain knowledge (angle-based control)
3. **Learned Policy**: Trained via RL (DQN, PPO, etc.)

## Demo 1: Random Policy (Baseline)

$$\pi_{\text{random}}(a|s) = \frac{1}{|\mathcal{A}|} = 0.5 \quad \forall s, a$$

**Expected Performance:** Poor - no learning or strategy

In [9]:
def run_random_policy(n_episodes=5, render=False):
    """Run CartPole with random action selection."""
    env = gym.make('CartPole-v1', render_mode='rgb_array' if render else None)
    episode_rewards = []
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        
        while True:
            action = env.action_space.sample()  # Random action
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            
            if terminated or truncated:
                break
        
        episode_rewards.append(total_reward)
        print(f"Episode {episode+1}: {total_reward:.0f} steps")
    
    env.close()
    return episode_rewards

print("=" * 50)
print("Random Policy Performance")
print("=" * 50)
random_rewards = run_random_policy(n_episodes=10)
print(f"\nAverage: {np.mean(random_rewards):.1f} ± {np.std(random_rewards):.1f} steps")

Random Policy Performance
Episode 1: 19 steps
Episode 2: 14 steps
Episode 3: 15 steps
Episode 4: 26 steps
Episode 5: 9 steps
Episode 6: 25 steps
Episode 7: 24 steps
Episode 8: 17 steps
Episode 9: 22 steps
Episode 10: 20 steps

Average: 19.1 ± 5.1 steps


In [10]:
# Visualize random policy with animation
print("Creating animation...\n")
anim_random, reward_random, frames_random = create_cartpole_animation(None, 'random', seed=123)
print(f"Random policy episode: {reward_random:.0f} steps")
HTML(anim_random.to_jshtml())

Creating animation...



DependencyNotInstalled: pygame is not installed, run `pip install "gymnasium[classic-control]"`

## Demo 2: Heuristic Policy

Use simple domain knowledge: **Push in the direction the pole is falling**

$$\pi_{\text{heuristic}}(s) = \begin{cases}
0 \text{ (left)} & \text{if } \theta < 0 \text{ (leaning left)} \\
1 \text{ (right)} & \text{if } \theta \geq 0 \text{ (leaning right)}
\end{cases}$$

Where $\theta = s_2$ is the pole angle.

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-top: 20px;">
    <strong>Intuition:</strong> Push the cart under the falling pole. Simple but reactive - doesn't anticipate motion!
</div>

In [ ]:
def heuristic_policy(state):
    """Simple heuristic: push in direction pole is falling."""
    pole_angle = state[2]
    # Basic reactive strategy - only considers current angle
    return 0 if pole_angle < 0 else 1

def run_heuristic_policy(n_episodes=5):
    """Run CartPole with heuristic policy."""
    env = gym.make('CartPole-v1')
    episode_rewards = []
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        
        while True:
            action = heuristic_policy(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            
            if terminated or truncated:
                break
        
        episode_rewards.append(total_reward)
        print(f"Episode {episode+1}: {total_reward:.0f} steps")
    
    env.close()
    return episode_rewards

print("=" * 50)
print("Heuristic Policy Performance")
print("=" * 50)
heuristic_rewards = run_heuristic_policy(n_episodes=10)
print(f"\nAverage: {np.mean(heuristic_rewards):.1f} ± {np.std(heuristic_rewards):.1f} steps")

In [ ]:
# Visualize heuristic policy with animation
print("Creating animation...\n")
anim_heuristic, reward_heuristic, frames_heuristic = create_cartpole_animation(
    heuristic_policy, 'heuristic', seed=123
)
print(f"Heuristic policy episode: {reward_heuristic:.0f} steps")
HTML(anim_heuristic.to_jshtml())

## Demo 3: Learned Policy (DQN)

**Deep Q-Network (DQN)**: Learns optimal action-values $Q^*(s,a)$

$$Q^*(s,a) = \max_\pi \mathbb{E}\left[\sum_{t=0}^\infty \gamma^t R_{t+1} \mid S_0=s, A_0=a, \pi\right]$$

Then choose actions greedily:
$$\pi^*(s) = \arg\max_a Q^*(s,a)$$

<div style="background-color: #d4edda; padding: 15px; border-left: 5px solid #28a745; margin-top: 20px;">
    <strong>✓ Next Week:</strong> We'll cover value functions, Q-learning, and the Bellman equation in detail.
</div>

In [ ]:
print("Training DQN agent...")
print("(This will take ~2-3 minutes)\n")

# Create vectorized environment
vec_env = make_vec_env('CartPole-v1', n_envs=4)

# Train DQN
model = DQN(
    'MlpPolicy',  # Multi-layer perceptron policy
    vec_env,
    learning_rate=1e-3,
    buffer_size=50000,
    learning_starts=2000,
    batch_size=64,
    tau=1.0,
    gamma=0.99,
    train_freq=4,
    target_update_interval=1000,
    verbose=0
)

model.learn(total_timesteps=100000)
print("✓ Training complete!\n")

In [ ]:
# Evaluate trained policy
def run_trained_policy(model, n_episodes=5):
    """Run CartPole with trained DQN policy."""
    env = gym.make('CartPole-v1')
    episode_rewards = []
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        
        while True:
            action, _ = model.predict(state, deterministic=True)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            
            if terminated or truncated:
                break
        
        episode_rewards.append(total_reward)
        print(f"Episode {episode+1}: {total_reward:.0f} steps")
    
    env.close()
    return episode_rewards

print("=" * 50)
print("Trained DQN Policy Performance")
print("=" * 50)
trained_rewards = run_trained_policy(model, n_episodes=10)
print(f"\nAverage: {np.mean(trained_rewards):.1f} ± {np.std(trained_rewards):.1f} steps")

In [ ]:
# Visualize trained DQN policy with animation
print("Creating animation...\n")
anim_dqn, reward_dqn, frames_dqn = create_cartpole_animation(model, 'trained', seed=123, max_frames=500)
print(f"DQN policy episode: {reward_dqn:.0f} steps")
HTML(anim_dqn.to_jshtml())

## Policy Comparison

Let's compare all three approaches:

In [ ]:
# Visualization of policy comparison
fig, ax = plt.subplots(figsize=(10, 6))

policies = ['Random', 'Heuristic', 'DQN (Learned)']
all_rewards = [random_rewards, heuristic_rewards, trained_rewards]
colors = ['#e74c3c', '#f39c12', '#27ae60']

positions = np.arange(len(policies))
bp = ax.boxplot(all_rewards, positions=positions, widths=0.6, 
                patch_artist=True, showmeans=True)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(policies)
ax.set_ylabel('Episode Length (steps)', fontsize=12)
ax.set_title('CartPole Performance: Policy Comparison', fontsize=14, fontweight='bold')
ax.axhline(y=500, color='green', linestyle='--', alpha=0.3, label='Max (500)')
ax.grid(axis='y', alpha=0.3)
ax.legend()

# Add mean values as text
for i, rewards in enumerate(all_rewards):
    mean_val = np.mean(rewards)
    ax.text(i, mean_val + 20, f'{mean_val:.1f}', 
            ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "=" * 60)
print("Summary Statistics")
print("=" * 60)
for policy, rewards in zip(policies, all_rewards):
    print(f"{policy:15s}: {np.mean(rewards):6.1f} ± {np.std(rewards):5.1f} steps")
print("=" * 60)

## Key Observations

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-top: 20px;">

<div style="background-color: #ffe6e6; padding: 15px; border-radius: 10px;">
    <h4 style="color: #e74c3c;">🎲 Random Policy</h4>
    <ul>
        <li>No learning or strategy</li>
        <li>High variance</li>
        <li>Poor average performance</li>
        <li><strong>Baseline for comparison</strong></li>
    </ul>
</div>

<div style="background-color: #fff8e6; padding: 15px; border-radius: 10px;">
    <h4 style="color: #f39c12;">🧠 Heuristic Policy</h4>
    <ul>
        <li>Uses simple domain knowledge</li>
        <li>Better than random</li>
        <li>Reactive, not predictive</li>
        <li><strong>Shows limits of hand-crafted rules</strong></li>
    </ul>
</div>

<div style="background-color: #e6f7e6; padding: 15px; border-radius: 10px; grid-column: 1 / -1;">
    <h4 style="color: #27ae60;">🤖 Learned Policy (DQN)</h4>
    <ul style="display: inline-block; margin: 0;">
        <li>Discovers optimal strategy through experience</li>
        <li>Consistently achieves maximum performance</li>
        <li>No manual feature engineering needed</li>
        <li><strong>Learns what humans might miss!</strong></li>
    </ul>
</div>

</div>

In [ ]:
# Create side-by-side comparison visualization
from matplotlib.gridspec import GridSpec

def create_comparison_frame():
    """Create a single frame comparing all three policies."""
    fig = plt.figure(figsize=(16, 5))
    gs = GridSpec(1, 3, figure=fig, wspace=0.3)
    
    # Create environments with same seed for fair comparison
    envs = [gym.make('CartPole-v1', render_mode='rgb_array') for _ in range(3)]
    states = [env.reset(seed=456)[0] for env in envs]
    
    policies = [(None, 'random'), (heuristic_policy, 'heuristic'), (model, 'trained')]
    titles = ['Random Policy', 'Heuristic Policy', 'Trained DQN']
    colors = ['#e74c3c', '#f39c12', '#27ae60']
    
    max_steps = 300
    all_frames = [[] for _ in range(3)]
    
    # Collect frames from all policies simultaneously
    for step in range(max_steps):
        all_done = True
        for i, ((policy_fn, policy_name), env, state) in enumerate(zip(policies, envs, states)):
            frame = env.render()
            all_frames[i].append(frame)
            
            if policy_name == 'random':
                action = env.action_space.sample()
            elif policy_name == 'heuristic':
                action = policy_fn(state)
            else:
                action, _ = policy_fn.predict(state, deterministic=True)
            
            next_state, _, terminated, truncated, _ = env.step(action)
            
            if not (terminated or truncated):
                all_done = False
                states[i] = next_state
        
        if all_done:
            break
    
    for env in envs:
        env.close()
    
    # Create animation
    axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
    for ax in axes:
        ax.axis('off')
    
    imgs = []
    for i, (ax, frames, title, color) in enumerate(zip(axes, all_frames, titles, colors)):
        img = ax.imshow(frames[0])
        imgs.append(img)
        ax.set_title(f'{title}\nSteps: {len(frames)}', 
                     fontsize=12, fontweight='bold', color=color, pad=10)
    
    def animate(frame_idx):
        for i, (img, frames) in enumerate(zip(imgs, all_frames)):
            if frame_idx < len(frames):
                img.set_array(frames[frame_idx])
        return imgs
    
    max_frames = max(len(frames) for frames in all_frames)
    anim = animation.FuncAnimation(
        fig, animate, frames=max_frames,
        interval=50, blit=True, repeat=True
    )
    
    plt.close(fig)
    return anim

print("Creating side-by-side comparison animation...\n")
comparison_anim = create_comparison_frame()
HTML(comparison_anim.to_jshtml())

## 
Episode Dynamics: State Trajectory

Let's visualize how the **state evolves** during an episode with different policies:

In [ ]:
def collect_trajectory(policy_fn, policy_name, seed=42):
    """Collect state trajectory for visualization."""
    env = gym.make('CartPole-v1')
    state, _ = env.reset(seed=seed)
    
    trajectory = {'states': [state], 'actions': [], 'rewards': []}
    
    for _ in range(500):  # Max episode length
        if policy_name == 'random':
            action = env.action_space.sample()
        elif policy_name == 'heuristic':
            action = policy_fn(state)
        else:  # trained
            action, _ = policy_fn.predict(state, deterministic=True)
        
        trajectory['actions'].append(action)
        state, reward, terminated, truncated, _ = env.step(action)
        trajectory['states'].append(state)
        trajectory['rewards'].append(reward)
        
        if terminated or truncated:
            break
    
    env.close()
    return np.array(trajectory['states']), np.array(trajectory['actions']), trajectory['rewards']

# Collect trajectories
states_rand, actions_rand, rewards_rand = collect_trajectory(None, 'random', seed=10)
states_heur, actions_heur, rewards_heur = collect_trajectory(heuristic_policy, 'heuristic', seed=10)
states_dqn, actions_dqn, rewards_dqn = collect_trajectory(model, 'trained', seed=10)

In [ ]:
# Plot state trajectories
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('State Evolution During Episode', fontsize=16, fontweight='bold')

labels = ['Cart Position', 'Cart Velocity', 'Pole Angle (rad)', 'Angular Velocity']
colors_policy = ['#e74c3c', '#f39c12', '#27ae60']
policy_names = ['Random', 'Heuristic', 'DQN']

for idx, (ax, label) in enumerate(zip(axes.flat, labels)):
    # Plot each policy
    ax.plot(states_rand[:, idx], color=colors_policy[0], alpha=0.7, 
            label=f'Random ({len(states_rand)} steps)', linewidth=2)
    ax.plot(states_heur[:, idx], color=colors_policy[1], alpha=0.7,
            label=f'Heuristic ({len(states_heur)} steps)', linewidth=2)
    ax.plot(states_dqn[:, idx], color=colors_policy[2], alpha=0.7,
            label=f'DQN ({len(states_dqn)} steps)', linewidth=2)
    
    ax.set_xlabel('Time Step', fontsize=10)
    ax.set_ylabel(label, fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=8)
    
    # Add bounds for position and angle
    if idx == 0:  # Cart position
        ax.axhline(y=2.4, color='red', linestyle='--', alpha=0.3, label='Bounds')
        ax.axhline(y=-2.4, color='red', linestyle='--', alpha=0.3)
    elif idx == 2:  # Pole angle
        ax.axhline(y=0.209, color='red', linestyle='--', alpha=0.3, label='Fail threshold')
        ax.axhline(y=-0.209, color='red', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## Return and Discount Factor $\gamma$

The **return** $G_t$ is the cumulative discounted reward:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

Where $\gamma \in [0, 1]$ is the **discount factor**:

- $\gamma = 0$: Only immediate reward matters (myopic)
- $\gamma = 1$: All future rewards equally important (far-sighted)
- $\gamma \in (0.9, 0.99)$: Common in practice (balance)

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-top: 20px;">
    <strong>Why discount?</strong>
    <ul>
        <li>Mathematical convenience (ensures convergence)</li>
        <li>Uncertainty about far future</li>
        <li>Preference for immediate rewards</li>
    </ul>
</div>

In [ ]:
# Visualize effect of discount factor
def compute_returns(rewards, gamma):
    """Compute discounted returns."""
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

# Use DQN trajectory (successful episode)
gammas = [0.9, 0.95, 0.99, 1.0]

fig, ax = plt.subplots(figsize=(12, 6))

for gamma in gammas:
    returns = compute_returns(rewards_dqn, gamma)
    ax.plot(returns, label=f'γ = {gamma}', linewidth=2, alpha=0.8)

ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Return $G_t$', fontsize=12)
ax.set_title('Effect of Discount Factor on Returns', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInitial Returns for Different γ:")
for gamma in gammas:
    returns = compute_returns(rewards_dqn, gamma)
    print(f"  γ = {gamma:4.2f}: G_0 = {returns[0]:7.2f}")

## Connection to EDA Problems

<div style="background-color: #fff3cd; padding: 20px; border-radius: 10px;">

### CartPole MDP ↔ EDA Problems

| CartPole | Boolean SAT | Technology Mapping | Placement |
|----------|-------------|-------------------|------------|
| **State**: Cart & pole config | Partial assignment | Current netlist mapping | Placed gates & remaining |
| **Action**: Push L/R | Assign variable T/F | Select library gate | Place gate at location |
| **Reward**: +1 per step | -conflicts, +satisfied clauses | -area, -delay | -wirelength, -overlap |
| **Termination**: Pole falls | SAT/UNSAT reached | All gates mapped | All gates placed |

</div>

<div style="background-color: #d4edda; padding: 15px; border-left: 5px solid #28a745; margin-top: 20px;">
    <strong>Key Insight:</strong> Like CartPole, EDA problems involve sequential decisions where each choice affects future options. RL can learn policies that traditional heuristics might miss!
</div>

## Summary: MDP Components

<div style="font-size: 0.9em;">

| Component | Symbol | CartPole Example | Purpose |
|-----------|---------|-----------------|----------|
| **State Space** | $\mathcal{S}$ | 4D continuous: $[x, \dot{x}, \theta, \dot{\theta}]$ | Describes world configuration |
| **Action Space** | $\mathcal{A}$ | Discrete: $\{0, 1\}$ (left, right) | Choices available to agent |
| **Transition** | $P(s'|s,a)$ | Physics equations (deterministic) | How actions change state |
| **Reward** | $R(s,a,s')$ | +1 per step, 0 at termination | Feedback signal for learning |
| **Discount** | $\gamma$ | 0.99 (typical) | Balances immediate vs future |
| **Policy** | $\pi(a|s)$ | Random / Heuristic / Learned | Agent's decision strategy |
| **Return** | $G_t$ | $\sum_{k=0}^T \gamma^k R_{t+k+1}$ | Cumulative discounted reward |

</div>

<div style="background-color: #065a82; color: white; padding: 15px; border-radius: 10px; margin-top: 20px; text-align: center;">
    <strong>Goal of RL:</strong> Find optimal policy $\pi^*$ that maximizes expected return
    $$\pi^* = \arg\max_\pi \mathbb{E}_{\pi}[G_t | S_t = s]$$
</div>

## Next Week: Value Functions

We'll build on these MDP foundations to introduce:

- **State-value function** $V^\pi(s)$: Expected return from state $s$
- **Action-value function** $Q^\pi(s,a)$: Expected return from $(s,a)$ pair  
- **Bellman equations**: Recursive relationships for value functions
- **Optimal value functions**: $V^*(s)$ and $Q^*(s,a)$
- **Policy improvement**: How to get better policies

<div style="background-color: #e8f4f8; padding: 20px; border-radius: 10px; margin-top: 20px;">
    <h4 style="margin-top: 0;">Preview Questions:</h4>
    <ol>
        <li>How do we <em>measure</em> how good a state is?</li>
        <li>How do we <em>compare</em> different actions?</li>
        <li>How do we <em>improve</em> our policy systematically?</li>
    </ol>
    <p style="margin-bottom: 0;"><strong>These are the questions value functions answer!</strong></p>
</div>

## Homework / Exploration

<div style="background-color: #fff3cd; padding: 20px; border-radius: 10px;">

### Suggested Activities:

1. **Experiment with different heuristics**
   - Try using both angle AND velocity: `action = 0 if (angle + velocity) < 0 else 1`
   - Compare performance to the simple heuristic

2. **Analyze failure cases**
   - Run the random policy 50 times and plot the distribution of episode lengths
   - What are the most common failure points?

3. **Explore other Gym environments**
   - Try `MountainCar-v0` or `Acrobot-v1`
   - Identify the MDP components (states, actions, rewards)

4. **Read ahead**
   - Sutton & Barto Chapter 3: Finite MDPs
   - Think about: How would you define an MDP for your research problem?

</div>

<div style="text-align: center; padding: 40px; background-color: #065a82; color: white; border-radius: 10px;">
    <h1 style="margin-top: 0;">Questions?</h1>
    <p style="font-size: 1.2em; margin-bottom: 0;">See you next week for Value Functions!</p>
</div>

---

<div style="text-align: center; color: #666; margin-top: 30px;">
    <p>Reinforcement Learning for EDA | Spring 2025</p>
</div>